# Lesson 32 — Multi-Objective Optimization

## Learning objectives

1. Define Pareto dominance, (weak) efficiency, the Pareto front, and the ideal and nadir points.
2. Use weighted-sum scalarization, and explain why it can only reach *supported* points: it misses non-convex parts of the front.
3. Generate a front with the $\varepsilon$-constraint method, and read the multiplier of the $\varepsilon$ row as the local **trade-off rate** between objectives.
4. Use the other scalarizations in `discopt.mo`: augmented weighted Tchebycheff, Normal Boundary Intersection (NBI) and Normalized Normal Constraint (NNC). Know when each one works.
5. Compare fronts with quality indicators: hypervolume, IGD, spread and the additive $\varepsilon$-indicator.
6. Choose a compromise point from a front, and recover the decision it stands for.
7. Build a mixed-integer bi-objective model (cost vs emissions facility location) whose front is non-convex, so weighted sum fails.

**Prerequisites:** the basic track. You should be comfortable with LP/MILP/NLP modeling in `discopt` (Lessons 3, 5, 9) and with shadow prices (Lesson 4).

## 1. Dominance and the Pareto front

A **multi-objective program** has $k \ge 2$ objectives to minimize together:

$$\min_{x \in X} \; F(x) = \big(f_1(x), \dots, f_k(x)\big).$$

$\mathbb{R}^k$ has no natural total order, so "minimize a vector" needs a definition {cite:p}`ehrgott-2005,miettinen-1999`. For two objective vectors $a, b$:

- $a$ **dominates** $b$ ($a \prec b$) if $a_i \le b_i$ for every $i$ and $a_j < b_j$ for at least one $j$.
- $x^\star \in X$ is **efficient** (Pareto optimal) if no $x \in X$ has $F(x) \prec F(x^\star)$.
- $x^\star$ is **weakly efficient** if no $x$ has $F(x) < F(x^\star)$ in *every* component. Weakly efficient points can still be improved in one objective at no cost in the others, so they are poor answers to give a decision maker.
- The image $\{F(x^\star) : x^\star \text{ efficient}\}$ is the **Pareto front** (nondominated set).

Two reference vectors bound the front:

- **Ideal point** $z^\star_i = \min_{x\in X} f_i(x)$. It is usually not attainable.
- **Nadir point** $z^{N}_i = \max\{f_i(x) : x \text{ efficient}\}$, the worst value objective $i$ takes *on the front*. For $k = 2$ it can be read from the two lexicographic anchors. For $k \ge 3$ the payoff-table estimate can be wrong in either direction {cite:p}`miettinen-1999`.

A maximized objective is handled by minimizing $-f_i$. `discopt.mo` does this for you when you pass `senses=["min", "max", ...]`, and it reports values in the original senses.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import matplotlib.pyplot as plt
from discopt.mo import (
    weighted_sum, epsilon_constraint, weighted_tchebycheff,
    normal_boundary_intersection, normalized_normal_constraint,
    filter_nondominated, common_reference, hypervolume, igd, spread,
    epsilon_indicator, ParetoFront, ParetoPoint,
)

# Dominance on a finite set. filter_nondominated returns a boolean mask;
# `senses` is +1 for a minimized column and -1 for a maximized one.
pts = np.array([[1.0, 5.0], [2.0, 3.0], [2.0, 4.0], [3.0, 3.0], [4.0, 1.0], [5.0, 1.0]])
mask = filter_nondominated(pts)                      # both minimized
print("nondominated:", pts[mask].tolist())
print("ideal :", pts[mask].min(axis=0), "  nadir:", pts[mask].max(axis=0))

# Same data, but now the second column is a *benefit* to maximize.
mask_max = filter_nondominated(pts, senses=np.array([1.0, -1.0]))
print("min f1 / max f2 nondominated:", pts[mask_max].tolist())


In the first call, $(2,4)$ is dominated by $(2,3)$, $(3,3)$ by $(2,3)$, and $(5,1)$ by $(4,1)$. The last pair ties in $f_2$, and $(4,1)$ still dominates $(5,1)$: dominance needs a strict improvement in only *one* objective. In the second call, maximizing $f_2$ makes $(1,5)$ best in both criteria, so it dominates every other point and the front collapses to a single point. When that happens the objectives do not actually conflict.

Multi-objective solvers do not return "the" answer. They **scalarize**: they turn the vector problem into a family of single-objective problems, each of which `discopt` solves to (global) optimality, and the solutions of that family trace out the front {cite:p}`marler-arora-2004`. The rest of this lesson compares scalarizations by asking which efficient points each one *can* reach.

## 2. Weighted-sum scalarization

Fix weights $w$ on the simplex, $w_i \ge 0$ and $\sum_i w_i = 1$, and solve

$$\min_{x \in X} \; \sum_{i=1}^k w_i\, f_i(x).$$

Two facts {cite:p}`ehrgott-2005` (Ch. 3):

1. If every $w_i > 0$, every minimizer is efficient. If some $w_i = 0$, minimizers are only weakly efficient.
2. **Converse (convex case).** If $X$ is convex and every $f_i$ is convex, every *properly* efficient point minimizes the weighted sum for some $w > 0$.

The geometry: the level set $\{z : w^\top z = c\}$ is a line (a hyperplane when $k > 2$), and the weighted-sum optimum is where this line first touches the image $F(X)$ as $c$ decreases. A point that some such line touches is called **supported**. Fact 2 says that on a convex problem every efficient point is supported.

`weighted_sum` normalizes each objective to $[0, 1]$ using the ideal and nadir (`normalize=True`, the default), then sweeps an evenly spaced grid of weights. Here is the convex QP $f_1 = x^2 + y^2$, $f_2 = (x-2)^2 + (y-1)^2$. Its front is the segment from $(0,0)$ to $(2,1)$ in decision space, and in objective space it is $\sqrt{f_1} + \sqrt{f_2} = \sqrt5$.

In [ ]:
def build_convex_qp():
    m = dm.Model("biobj_qp")
    x = m.continuous("x", lb=-5, ub=5)
    y = m.continuous("y", lb=-5, ub=5)
    return m, [x**2 + y**2, (x - 2)**2 + (y - 1)**2]

m, objs = build_convex_qp()                     # scalarizers mutate the model:
front_ws_qp = weighted_sum(m, objs, n_weights=11)  # build a fresh one per sweep
print(front_ws_qp.summary())

a = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(5 * a**2, 5 * (1 - a)**2, "k-", lw=1, label=r"analytic front")
Z = front_ws_qp.objectives()
ax.scatter(Z[:, 0], Z[:, 1], c="C0", zorder=3, label="weighted sum (11 weights)")
ax.set_xlabel("$f_1$"); ax.set_ylabel("$f_2$"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()


On a convex front the weighted sum works, though uniform weights do **not** give uniformly spaced points. The points crowd where the front is strongly curved.

## 3. Why the weighted sum misses non-convex parts of the front

Now change the feasible set so that the front bends the other way. Minimize $f_1 = x$ and $f_2 = y$ over $\{(x,y) \in [0,1]^2 : x^2 + y^2 \ge 1\}$. The front is the quarter circle from $(1,0)$ to $(0,1)$, and it bulges *toward* the ideal $(0,0)$.

A line $w_1 f_1 + w_2 f_2 = c$ sliding down toward the origin touches this arc first at an **endpoint**. Every interior point of the arc lies above the chord between the anchors, so no weight can select it. Interior points of the arc are efficient but **unsupported**. This is not a flaw of the solver or the weight grid. It follows from the geometry, and refining the grid cannot fix it {cite:p}`marler-arora-2004,ehrgott-2005`.

In [ ]:
def build_concave():
    m = dm.Model("concave")
    x = m.continuous("x", lb=0, ub=1)
    y = m.continuous("y", lb=0, ub=1)
    m.subject_to(x**2 + y**2 >= 1.0)          # non-convex feasible set
    return m, [x, y]

m, objs = build_concave()
front_ws_cc = weighted_sum(m, objs, n_weights=11)
m, objs = build_concave()
front_eps_cc = epsilon_constraint(m, objs, n_points=11)
print("weighted sum   :", front_ws_cc.n, "points ->", front_ws_cc.objectives().round(3).tolist())
print("eps-constraint :", front_eps_cc.n, "points")


With 11 weights the weighted sum returns only the 2 anchors, while the $\varepsilon$-constraint method (next section) traces the whole arc. Non-convex fronts are common in practice. Feasible sets with integer variables (Section 6), non-convex physics and economies of scale all produce them.

## 4. The $\varepsilon$-constraint method and trade-off rates

Keep one objective and turn the others into constraints {cite:p}`ehrgott-2005,miettinen-1999`:

$$P(\varepsilon):\qquad \min_{x\in X} f_1(x) \quad\text{s.t.}\quad f_j(x) \le \varepsilon_j,\;\; j = 2,\dots,k.$$

**Completeness.** $x^\star$ is efficient if and only if it solves $P(\varepsilon)$ with $\varepsilon_j = f_j(x^\star)$ for every choice of the kept objective. In particular, any efficient point, supported or not, is *reachable* by some $\varepsilon$. Convexity is not needed. The price is that a single $P(\varepsilon)$ may return a point that is only weakly efficient, and that the grid of $\varepsilon$ values has to be fine enough to hit every point.

**AUGMECON.** `discopt.mo.epsilon_constraint` implements the augmented variant AUGMECON2 {cite:p}`mavrotas-2009`. It turns each row into an equality with a slack, $f_j(x) + s_j = \varepsilon_j$, and adds a tiny reward $-\delta \sum_j s_j / r_j$ to the objective. This guarantees strict efficiency. It also builds the $\varepsilon$ range from a *lexicographic* payoff table, and it skips $\varepsilon$ cells that the current slack proves would return the same point ("bypass").

**Trade-off rates from duals.** Let $z^\star(\varepsilon)$ be the optimal value of $P(\varepsilon)$ for $k = 2$, and let $\lambda \ge 0$ be the KKT multiplier of the row $f_2(x) \le \varepsilon$. The envelope theorem from Lesson 4 gives

$$\frac{d z^\star}{d\varepsilon} = -\lambda .$$

So $\lambda$ is the **local slope of the Pareto front**: the number of units of $f_1$ you must give up to improve $f_2$ by one unit. When $f_1$ is cost and $f_2$ is emissions, $\lambda$ is the marginal abatement cost. Duality ties this back to the weighted sum. At a supported point, the weights that select it satisfy $w_2 / w_1 = \lambda$ {cite:p}`miettinen-1999,Boyd2004`.

The LP below makes a product on three processes. $f_1$ is cost and $f_2$ is emissions. We read $\lambda$ three ways: from `r.constraint_duals`, from `r.gradient` of the $\varepsilon$ parameter, and from finite differences of $z^\star$.

In [ ]:
c_cost = np.array([2.0, 3.0, 5.0])     # cost per unit from process 1, 2, 3
e_rate = np.array([6.0, 3.0, 1.0])     # emissions per unit

m = do.Model("eps_lp")
x = m.continuous("x", shape=(3,), lb=0, ub=8)
eps = m.parameter("eps", 30.0)                       # epsilon as a Parameter
m.subject_to(x[0] + x[1] + x[2] >= 10, name="demand")
m.subject_to(x[0] + 2 * x[1] <= 12, name="capacity")
m.subject_to(e_rate[0] * x[0] + e_rate[1] * x[1] + e_rate[2] * x[2] <= eps, name="emis")
m.minimize(c_cost[0] * x[0] + c_cost[1] * x[1] + c_cost[2] * x[2])

grid = np.arange(18.0, 63.0, 3.0)
z, lam_dual, lam_grad = [], [], []
for ev in grid:
    eps.value = ev
    r = m.solve(sensitivity=True)
    z.append(r.objective)
    lam_dual.append(float(r.constraint_duals["emis"]))
    lam_grad.append(-float(r.gradient(eps)))
z = np.array(z)
lam_fd = -np.gradient(z, grid)                      # finite-difference slope
for row in zip(grid, z, lam_dual, lam_grad, lam_fd):
    print("eps=%5.1f  z*=%7.3f  lambda: dual=%5.2f  grad=%5.2f  fd=%5.2f" % row)


The three estimates agree away from breakpoints. $z^\star(\varepsilon)$ is piecewise linear and convex, as the front of an LP always is. Its slope jumps where the optimal basis changes ($\lambda$ steps from $1$ to $0.5$ to $0$). The centred finite difference straddles each kink, so it shows the average of the two slopes there. For $\varepsilon > 54$ the emissions row is slack and $\lambda = 0$: cutting the emissions allowance from a higher value down to 54 costs nothing. The cheapest plan emits 54, which is the cost anchor. Exactly at a kink such as $\varepsilon = 54$, $z^\star$ has no derivative, so the dual, the gradient and the finite difference can each report a different one-sided or averaged slope. At the other end, $P(\varepsilon)$ becomes infeasible below $\varepsilon = 14$, which is the emissions anchor. The sweep above starts at 18.

`discopt.mo.epsilon_constraint` automates the sweep. It computes the anchors, lays the grid over $[z^\star_2, z^N_2]$, and returns a `ParetoFront`:

In [ ]:
m = do.Model("eps_lp_mo")
x = m.continuous("x", shape=(3,), lb=0, ub=8)
m.subject_to(x[0] + x[1] + x[2] >= 10)
m.subject_to(x[0] + 2 * x[1] <= 12)
cost = c_cost[0] * x[0] + c_cost[1] * x[1] + c_cost[2] * x[2]
emis = e_rate[0] * x[0] + e_rate[1] * x[1] + e_rate[2] * x[2]
front_lp = epsilon_constraint(m, [cost, emis], n_points=8,
                              objective_names=["cost", "emissions"])
print(front_lp.summary())
print("incomplete?", front_lp.incomplete)   # True would mean a sub-solve failed

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(z, grid, "k-", lw=1, label=r"$z^\star(\varepsilon)$ from the manual sweep")
Z = front_lp.objectives()
ax.scatter(Z[:, 0], Z[:, 1], c="C1", zorder=3, label="epsilon_constraint")
ax.set_xlabel("cost $f_1$"); ax.set_ylabel(r"emissions $f_2$"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()


## 5. Other scalarizations in `discopt.mo`

All of the scalarizations below share one signature: `method(model, objectives, senses=..., objective_names=..., n_points | n_weights=..., **solve_kwargs)`. Each returns a `ParetoFront` whose dominated points have already been filtered out.

**Augmented weighted Tchebycheff** {cite:p}`steuer-choo-1983`. Measure the distance to the ideal point with a weighted $L_\infty$ norm, and add a small $L_1$ term:

$$\min_{x,t}\; t + \rho \sum_i w_i\, g_i(x) \quad\text{s.t.}\quad t \ge w_i\, g_i(x),\qquad g_i = \frac{f_i - z^\star_i}{z^N_i - z^\star_i}.$$

Every efficient point is the solution for *some* $w$, so the method covers non-convex fronts. The term $\rho > 0$ removes weakly efficient solutions. Passing `reference=` replaces the ideal $z^\star$ with an aspiration point that the decision maker supplies. This is the reference-point or "achievement" form, and it is how interactive methods let a user steer toward a region of the front.

**Normal Boundary Intersection (NBI)** {cite:p}`das-dennis-1998`. Take the convex hull of the anchors (CHIM), spread evenly spaced points $\Phi w$ along it, and from each one move along the quasi-normal toward the ideal until you hit the boundary of $F(X)$:

$$\max_{x,t}\; t \quad \text{s.t.}\quad \Phi w + t\,\hat n = g(x) - g^\star .$$

The equality constraint makes NBI give nearly *evenly spaced* points on smooth fronts. It is a geometric method, though. It can return dominated points on folded fronts, and the equality is usually infeasible when $x$ is integer.

**Normalized Normal Constraint (NNC)** {cite:p}`messac-2003`. Keep the same evenly spaced points on the utopia hyperplane, but replace NBI's equality with a half-space cut and minimize the last objective. It is more robust than NBI. It still needs a dominance filter afterwards.

In [ ]:
methods = [
    ("weighted sum", weighted_sum,                  dict(n_weights=11), "o"),
    ("AUGMECON2",    epsilon_constraint,            dict(n_points=11),  "s"),
    ("Tchebycheff",  weighted_tchebycheff,          dict(n_weights=11), "^"),
    ("NBI",          normal_boundary_intersection,  dict(n_points=11),  "D"),
    ("NNC",          normalized_normal_constraint,  dict(n_points=11),  "P"),
]
fronts_cc = {}
for name, fn, kw, _ in methods:
    m, objs = build_concave()
    fronts_cc[name] = fn(m, objs, **kw)

th = np.linspace(0, np.pi / 2, 201)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(np.cos(th), np.sin(th), "k-", lw=1, label="true front")
for name, _, _, mk in methods:
    Z = fronts_cc[name].objectives()
    ax.scatter(Z[:, 0], Z[:, 1], marker=mk, alpha=0.75, label=f"{name} ({fronts_cc[name].n})")
ax.set_xlabel("$f_1$"); ax.set_ylabel("$f_2$"); ax.set_aspect("equal")
ax.set_title("Concave front: only the weighted sum fails"); ax.legend(fontsize=8)
plt.tight_layout()


Look at the *spacing* too. AUGMECON2 places points evenly in $f_2$, the variable it constrains. Tchebycheff places them evenly in angle as seen from the ideal. NBI and NNC place them evenly along the chord. All four are complete on this continuous front.

## 6. A mixed-integer example: cost vs emissions facility location

Integer variables make the image $F(X)$ a **finite set of points**. Its nondominated points sit on a "staircase" that is almost never convex. Weighted sums reach only the supported points, the ones on the lower-left convex hull of the front. Every other nondominated point is unsupported, and weighted sums cannot find it {cite:p}`ehrgott-2005`.

The model is a single-sourcing capacitated facility location. There are 4 candidate sites and 6 customers. Site $j$ has fixed cost $\phi_j$, capacity $u_j$ and emission rate $e_j$ per unit served. The sites that are cheap to open are the dirty ones:

$$\begin{aligned}
\min\;& f_1 = \textstyle\sum_j \phi_j y_j + \tfrac12 \sum_{j,i} \mathrm{dist}_{ji}\, d_i\, x_{ji} && \text{(cost)}\\
\min\;& f_2 = \textstyle\sum_{j,i} e_j\, d_i\, x_{ji} && \text{(emissions)}\\
\text{s.t. }& \textstyle\sum_j x_{ji} = 1 \;\;\forall i,\qquad \sum_i d_i x_{ji} \le u_j\, y_j \;\;\forall j,\qquad x, y \text{ binary.}
\end{aligned}$$

The instance is small enough to **enumerate** every assignment. That gives the exact front to check each method against.

In [ ]:
import itertools

rng = np.random.default_rng(3)
nS, nC = 4, 6
sites, cust = rng.uniform(0, 10, (nS, 2)), rng.uniform(0, 10, (nC, 2))
dist = np.linalg.norm(sites[:, None] - cust[None], axis=2)      # (nS, nC)
d = rng.integers(5, 15, nC).astype(float)                        # demands
phi = np.array([30.0, 45.0, 60.0, 80.0])                         # fixed cost
e = np.array([3.0, 2.0, 1.2, 0.5])                               # emissions / unit
u = np.full(nS, 40.0)                                            # capacity

def build_fl():
    m = do.Model("facility_biobj")
    y = m.binary("y", shape=(nS,))
    x = m.binary("x", shape=(nS, nC))
    for i in range(nC):
        m.subject_to(sum(x[j, i] for j in range(nS)) == 1)
    for j in range(nS):
        m.subject_to(sum(d[i] * x[j, i] for i in range(nC)) <= u[j] * y[j])
    cost = (sum(phi[j] * y[j] for j in range(nS))
            + sum(0.5 * dist[j, i] * d[i] * x[j, i] for j in range(nS) for i in range(nC)))
    emis = sum(e[j] * d[i] * x[j, i] for j in range(nS) for i in range(nC))
    return m, [cost, emis]

# Exact front by enumeration (4^6 = 4096 assignments; open = any site used).
allpts = []
for a in itertools.product(range(nS), repeat=nC):
    a = np.array(a)
    load = np.bincount(a, weights=d, minlength=nS)
    if np.all(load <= u):
        allpts.append((phi @ (load > 0) + 0.5 * dist[a, np.arange(nC)] @ d, e[a] @ d))
allpts = np.unique(np.round(allpts, 6), axis=0)
true_front = allpts[filter_nondominated(allpts)]
true_front = true_front[np.argsort(true_front[:, 0])]
print(len(allpts), "distinct (cost, emissions) outcomes;", len(true_front), "nondominated:")
print(true_front.round(2))


We run three sweeps on a fresh model each time. By default (`warm_start=True`) each sub-MILP starts from the previous point's solution, which seeds its incumbent: it changes where the search starts, never what the solve certifies. Section 9 lists the checks to run on any front.

In [ ]:
names = ["cost", "emissions"]
m, objs = build_fl()
fl_ws  = weighted_sum(m, objs, n_weights=21, objective_names=names)
m, objs = build_fl()
fl_eps = epsilon_constraint(m, objs, n_points=21, objective_names=names)
m, objs = build_fl()
fl_tch = weighted_tchebycheff(m, objs, n_weights=21, objective_names=names)

def recovered(front):
    Z = front.objectives()
    return sum(np.any(np.all(np.isclose(Z, p, atol=1e-4), axis=1)) for p in true_front)

for name, F in [("weighted sum", fl_ws), ("AUGMECON2", fl_eps), ("Tchebycheff", fl_tch)]:
    print(f"{name:13s}: {F.n:2d} points, {recovered(F)}/{len(true_front)} true Pareto points, "
          f"incomplete={F.incomplete}")

# Which true points are *supported* (optimal for some weighted sum)?
Nf = (true_front - true_front.min(0)) / np.ptp(true_front, axis=0)
lams = np.linspace(0, 1, 2001)
scores = lams[:, None] * Nf[:, 0] + (1 - lams[:, None]) * Nf[:, 1]       # (L, n)
supported = np.any(scores <= scores.min(axis=1, keepdims=True) + 1e-12, axis=0)
print("supported points:", int(supported.sum()), " unsupported:", int((~supported).sum()))


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(allpts[:, 0], allpts[:, 1], s=6, c="lightgray", label="all outcomes")
ax.step(true_front[:, 0], true_front[:, 1], where="post", c="k", lw=0.8)
hull = true_front[supported]
ax.plot(hull[:, 0], hull[:, 1], "C0--", lw=1, label="convex hull (supported)")
Zt, Zw = fl_tch.objectives(), fl_ws.objectives()
ax.scatter(Zt[:, 0], Zt[:, 1], s=70, facecolors="none", edgecolors="C2", lw=1.5,
           label="Tchebycheff / AUGMECON2")
ax.scatter(Zw[:, 0], Zw[:, 1], marker="x", s=60, c="C0", label="weighted sum")
ax.set_xlim(190, 275); ax.set_ylim(35, 145)
ax.set_xlabel("cost"); ax.set_ylabel("emissions"); ax.legend(fontsize=8)
ax.set_title("Facility location: unsupported points lie above the hull")
plt.tight_layout()


The weighted sum finds only the points on the dashed hull. AUGMECON2 and Tchebycheff recover the whole front. With 21 grid values the $\varepsilon$ grid happened to be fine enough here. On a front with close-together points you must refine the grid (Exercise 5).

> **If your run prints `incomplete=True` for AUGMECON2.** On some discopt development builds (0.9.1.dev0), the HiGHS route can return binaries that are integral only to about $10^{-7}$. The first lexicographic solve then reports emissions as 136.99999884 instead of 137, and that value becomes an $\varepsilon$ bound. The sub-MILP at that bound is then rejected because its incumbent violates a row by about $3\times 10^{-6}$, so the sweep reports 8 of 9 points. This is a solver tolerance issue, not a property of the method. It is also exactly why checklist item 2 below says to look at `front.incomplete` and `front.incomplete_cells` before trusting a front.

The geometric methods have trouble on discrete fronts. NBI asks for $g(x)$ to lie *exactly* on a ray, and integer $x$ can almost never satisfy that. Both NBI and NNC also build their anchors from a *simple* payoff table, which can keep a weakly efficient anchor:

In [ ]:
m, objs = build_fl()
fl_nbi = normal_boundary_intersection(m, objs, n_points=11, objective_names=names)
m, objs = build_fl()
fl_nnc = normalized_normal_constraint(m, objs, n_points=11, objective_names=names)
print("NBI points:\n", fl_nbi.objectives().round(2))
print("NNC points:\n", fl_nnc.objectives().round(2))
truly_efficient = lambda F: [bool(np.any(np.all(np.isclose(true_front, p, atol=1e-4), axis=1)))
                             for p in F.objectives()]
print("NBI efficient?", truly_efficient(fl_nbi))
print("NNC efficient?", truly_efficient(fl_nnc))


NBI returns only the two anchors. Its emissions anchor costs about $339$, while the efficient point with the same emissions costs $264$: minimizing emissions alone has alternative optima, and the anchor solve kept an expensive one. NNC also reports points that are dominated by Pareto points it never found. A front's own dominance filter cannot catch this, because it compares points only against each other. **Rule:** for mixed-integer problems, use AUGMECON2 (with its lexicographic payoff) or Tchebycheff.

## 7. Quality indicators

To compare fronts we need numbers. `discopt.mo` provides four indicators {cite:p}`zitzler-deb-thiele-2000`:

| indicator | definition | better |
|---|---|---|
| hypervolume $HV(A; r)$ | measure of the region dominated by $A$ and bounded by a reference point $r$ | larger |
| IGD$(A; R)$ | mean distance from each point of a reference front $R$ to its nearest point in $A$ | smaller |
| spread | coefficient of variation of the gaps between neighbouring points | smaller (more even) |
| $I_\varepsilon(A, B)$ | smallest shift $\varepsilon$ such that $A - \varepsilon$ weakly dominates every point of $B$ | smaller |

Hypervolume is the only one of these that is **Pareto compliant**: if $A$ dominates $B$, then $HV(A) > HV(B)$. Its value depends on $r$, though. `hypervolume(front)` with no reference uses a reference built from that front, so two fronts get different references. **Always pass a shared `common_reference(...)` when comparing.** IGD and $I_\varepsilon$ need a reference front, which here is the enumerated exact front. All of the indicators are computed in raw objective units, so normalize the objectives first when their scales differ a lot.

In [ ]:
ref = common_reference(fl_ws, fl_eps, fl_tch)
exact = ParetoFront(points=[ParetoPoint(x={}, objectives=p, status="enumerated", wall_time=0.0)
                            for p in true_front],
                    method="enumeration", objective_names=names, senses=["min", "min"])
print("shared reference point:", ref.round(3))
print(f"{'method':13s} {'HV':>9s} {'IGD':>7s} {'spread':>7s} {'I_eps':>7s}")
for name, F in [("exact", exact), ("weighted sum", fl_ws), ("AUGMECON2", fl_eps), ("Tchebycheff", fl_tch)]:
    print(f"{name:13s} {F.hypervolume(reference=ref):9.2f} {igd(F, exact):7.3f} "
          f"{spread(F):7.3f} {epsilon_indicator(F, exact):7.3f}")


The weighted-sum front has a lower hypervolume and a positive IGD and $I_\varepsilon$, because points are missing. Its spread is *better*, since a few hull points are more evenly spaced than the whole staircase. Spread on its own rewards fronts that leave points out, so never rank fronts by spread alone.

## 8. Choosing a compromise

A front is a menu, and someone still has to pick from it. Common *a posteriori* rules {cite:p}`miettinen-1999,marler-arora-2004` all start by normalizing with the ideal and nadir, $\tilde f_i = (f_i - z^\star_i)/(z^N_i - z^\star_i) \in [0,1]$:

- **Compromise programming.** Pick the point closest to the ideal, $\min \|\tilde F(x)\|_p$. With $p = 2$ this is Euclidean distance. With $p = \infty$ it is equal-weight Tchebycheff, which you can solve directly with `weighted_tchebycheff(..., weights=[[0.5, 0.5]])`, even for unsupported points.
- **Knee point.** Pick the point farthest below the chord joining the anchors, where giving up a little of one objective buys a lot of the other.
- **Trade-off threshold.** Walk along the front until the marginal rate ($\lambda$ from Section 4, or $-\Delta f_1/\Delta f_2$ between neighbouring points) exceeds what the decision maker will pay, such as a carbon price.

A `ParetoPoint` stores the full decision, `point.x[var_name]`, so the chosen compromise maps straight back to the decision it represents.

In [ ]:
F = fl_eps
Z = F.objectives()
Zn = (Z - F.ideal) / (F.nadir - F.ideal)

i_l2   = int(np.argmin(np.linalg.norm(Zn, axis=1)))
i_linf = int(np.argmin(Zn.max(axis=1)))
i_knee = int(np.argmax((1 - Zn[:, 0] - Zn[:, 1]) / np.sqrt(2)))
for label, i in [("closest to ideal (L2)", i_l2), ("min-max (L_inf)", i_linf), ("knee", i_knee)]:
    open_sites = np.flatnonzero(F.points[i].x["y"] > 0.5).tolist()
    print(f"{label:22s} cost={Z[i, 0]:7.2f}  emissions={Z[i, 1]:6.1f}  open sites={open_sites}")

# Marginal abatement cost between consecutive Pareto points (discrete analogue of lambda).
order = np.argsort(Z[:, 0])
Zs = Z[order]
mac = -np.diff(Zs[:, 0]) / np.diff(Zs[:, 1])
print("cost per unit of emissions avoided, moving along the front:", mac.round(2))


Different rules pick different points, and that is expected. Choosing a compromise is a judgment about values, and the optimizer cannot make it. The optimizer's job is to make the menu **complete** (no missing unsupported points) and **honest** (every point truly efficient), so that the judgment is an informed one. Notice that the discrete trade-off rate is *not* monotone. A non-monotone rate is the signature of a non-convex front: a decision maker paying a fixed carbon price would skip whole stretches of it.

## 9. Practical checklist

1. **Fresh model per sweep.** Scalarizers add auxiliary parameters, variables and constraints to the model you pass. Build it with a function, as above.
2. **Check `front.incomplete`.** A sub-solve that hits a limit leaves a hole. `front.incomplete_cells` lists those holes.
3. **Check the anchors.** If minimizing one objective alone has alternative optima, an anchor can be weakly efficient (NBI above). AUGMECON2's lexicographic payoff avoids this.
4. **Resolve the grid.** AUGMECON2 is complete only if consecutive $\varepsilon$ values are closer together than consecutive front points. For integer-valued objectives a step of 1 is exact.
5. **Validate small instances.** Enumerate, or run a very fine grid, and compare with an indicator before trusting a heuristic choice of grid on the full-size model.

## References

{cite:p}`ehrgott-2005` (Ch. 3–4, scalarization; Ch. 8–9, multi-objective integer programming); {cite:p}`miettinen-1999` (Part II, methods; trade-offs and KKT); {cite:p}`marler-arora-2004` (engineering survey); {cite:p}`mavrotas-2009` (AUGMECON); {cite:p}`steuer-choo-1983` (Tchebycheff); {cite:p}`das-dennis-1998` (NBI); {cite:p}`messac-2003` (NNC); {cite:p}`zitzler-deb-thiele-2000` (indicators); {cite:p}`Boyd2004` (duality and the envelope theorem).